In [ ]:
# %% [markdown]
# # 02 — Customer Segmentation (RFM + K-Means + DBSCAN)
# ## RetailPulse — Zidio Development | March 2026


In [ ]:
# %% [markdown]
# ## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

In [ ]:
# %%[markdown]
# ## Load Cleaned Data

In [ ]:
sales = pd.read_parquet('../data/processed/sales_clean.parquet')
print(f"Sales: {len(sales):,} rows")
print(f"Date range: {sales['date'].min()} to {sales['date'].max()}")

In [ ]:
# %%[markdown]
# ## Compute RFM Features

In [ ]:
snapshot_date = sales['date'].max() + pd.Timedelta(days=1)

rfm = sales.groupby('customer_id').agg(
    Recency=('date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('transaction_id', 'nunique'),
    Monetary=('revenue', 'sum')
).reset_index()

print(f"RFM shape: {rfm.shape}")
print(rfm.describe())

In [ ]:
# %%[markdown]
# ## RFM Scoring (1-5 Quantiles)

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
print(rfm.head())

In [ ]:
# %%[markdown]
# ## Optimal K — Elbow + Silhouette

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = rfm[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertias, 'bo-')
ax1.set_title('Elbow Method')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia')
ax2.plot(K_range, silhouettes, 'rs-')
ax2.set_title('Silhouette Score')
ax2.set_xlabel('K'); ax2.set_ylabel('Score')
plt.tight_layout()
plt.savefig('../reports/elbow_silhouette.png', bbox_inches='tight')
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Optimal K (silhouette): {best_k}")

In [ ]:
# %%[markdown]
# ## K-Means Clustering (k=6)

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
rfm['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

segment_names = {
    0: 'Champions', 1: 'Loyal Customers', 2: 'Potential Loyalists',
    3: 'At-Risk Customers', 4: 'Lost Customers', 5: 'New Customers'
}
rfm['Segment_Name'] = rfm['KMeans_Cluster'].map(segment_names)

cluster_summary = rfm.groupby('KMeans_Cluster')[features].mean().round(2)
print("Cluster Summary:")
print(cluster_summary)
print(f"\nSegment distribution:")
print(rfm['Segment_Name'].value_counts())

In [ ]:
# %%[markdown]
# ## DBSCAN Clustering

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
rfm['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)
n_noise = (rfm['DBSCAN_Cluster'] == -1).sum()
print(f"DBSCAN clusters: {rfm['DBSCAN_Cluster'].nunique() - 1} (+ noise)")
print(f"Noise points: {n_noise} ({n_noise/len(rfm)*100:.1f}%)")

In [ ]:
# %%[markdown]
# ## 3D RFM Visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(rfm['Recency'], rfm['Frequency'], rfm['Monetary'],
                     c=rfm['KMeans_Cluster'], cmap='tab10', alpha=0.6, s=20)
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary (£)')
plt.title('Customer Segments — 3D RFM Space')
plt.colorbar(scatter, label='Cluster')
plt.savefig('../reports/rfm_3d_clusters.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Save Results

In [ ]:
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print(f"Saved rfm_segments.csv with {len(rfm)} customers")

# Save segment profiles for dashboard
segment_profiles = rfm.groupby('Segment_Name').agg(
    count=('customer_id', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
).round(2).reset_index()
segment_profiles.to_csv('../data/processed/segment_profiles.csv', index=False)
print("Saved segment_profiles.csv")